In [ ]:
# 安裝套件
%pip install --upgrade pip
%pip install -r requirements.txt

In [ ]:
# 錄製影片素材
import cv2
import os
import time
from datetime import datetime
from dotenv import load_dotenv
from IPython.display import clear_output, display, Image

load_dotenv()

os.environ["OPENCV_FFMPEG_CAPTURE_OPTIONS"] = "timeout;10000000|rw_timeout;10000000"

# 參數設定
CCTV_URL = os.environ["CCTV_URL"]
RECORD_SECONDS = 1200
FPS = 20.0

# 建立儲存路徑
save_path = "origin.mp4"

# 開啟串流
cap = cv2.VideoCapture(CCTV_URL)

if not cap.isOpened():
    print(f"❌ 無法開啟串流 {CCTV_URL}")
else:
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  or 1280
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 720
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(save_path, fourcc, FPS, (width, height))

    print(f"🚀 開始錄製純影片...")
    print(f"📂 儲存路徑: {save_path}")
    print(f"⏱️  預計時長: {RECORD_SECONDS} 秒")

    TOTAL_FRAMES = int(RECORD_SECONDS * FPS)
    frame_count = 0

    try:
        while frame_count < TOTAL_FRAMES:
            ret, frame = cap.read()
            if not ret:
                print("⚠️ 串流暫時中斷，重試中...")
                time.sleep(1)
                continue

            out.write(frame)
            frame_count += 1

            if frame_count % int(FPS) == 0:
                clear_output(wait=True)
                pct = (frame_count / TOTAL_FRAMES) * 100
                print(f"🔴 錄製中：{pct:.1f}% [{frame_count}/{TOTAL_FRAMES}]")
                resize_frame = cv2.resize(frame, (640, 360))
                _, encoded_img = cv2.imencode('.jpg', resize_frame)
                display(Image(data=encoded_img.tobytes()))

    except KeyboardInterrupt:
        print("\n🛑 手動中斷錄製")

    finally:
        cap.release()
        out.release()
        print(f"✅ 錄製結束，存儲於: {os.path.abspath(save_path)}")

In [ ]:
# 切分影片並上傳 Roboflow，將標註、時間存入 SQLite
import cv2
import os
import json
import sqlite3
from dotenv import load_dotenv
from inference_sdk import InferenceHTTPClient
from inference_sdk.webrtc import VideoFileSource, StreamConfig, VideoMetadata

load_dotenv()

# --- 1. 影片切片配置 (每 10 分鐘一段) ---
INPUT_VIDEO = "origin.mp4"
DB_PATH = "roboflow_data.db"
CHUNK_DURATION_MIN = 10
CHUNK_DURATION_SEC = CHUNK_DURATION_MIN * 60

def split_video(video_path, chunk_sec):
    """將大影片切片成多個 10 分鐘的小影片，返回小影片路徑清單"""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames_per_chunk = int(fps * chunk_sec)
    
    chunk_files = []
    chunk_index = 0
    frame_count = 0
    out = None
    
    print("正在進行影片本地切片...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        if frame_count % frames_per_chunk == 0:
            if out:
                out.release()
            chunk_file = f"temp_chunk_{chunk_index}.mp4"
            out = cv2.VideoWriter(chunk_file, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
            chunk_files.append(chunk_file)
            chunk_index += 1
            
        out.write(frame)
        frame_count += 1
        
    if out:
        out.release()
    cap.release()
    print(f"切片完成，共切成 {len(chunk_files)} 段。")
    return chunk_files, fps

# --- 2. 初始化 SQLite (每次執行重置) ---
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.executescript("""
DROP TABLE IF EXISTS frames;
CREATE TABLE frames (
    frame_id         INTEGER PRIMARY KEY,
    pts_ms           REAL,
    predictions_json TEXT
);
""")
conn.commit()
print(f"SQLite 已重置：{os.path.abspath(DB_PATH)}")

# 執行切片
chunk_files, original_fps = split_video(INPUT_VIDEO, CHUNK_DURATION_SEC)

# --- 3. 全局狀態初始化 ---
global_frame_offset = 0       # 用於累加不同片段的影格 ID
global_time_offset_ms = 0.0   # 用於累加不同片段的時間戳

DATA_OUTPUTS = ["predictions"]  # 只取預測標註，不請求/解碼影像
debug_samples_left = 3          # 印出前幾個非空 payload 的結構，方便確認欄位

# Roboflow 工作流設定：預設直接使用作者已發布的公開工作流。
# 進階使用者若要改用自己的工作流，可在 .env 設定 ROBOFLOW_WORKSPACE / ROBOFLOW_WORKFLOW 覆寫。
ROBOFLOW_WORKSPACE = os.environ.get("ROBOFLOW_WORKSPACE", "cowrider2018")
ROBOFLOW_WORKFLOW  = os.environ.get("ROBOFLOW_WORKFLOW", "carcounter")

# --- 4. 初始化 Roboflow 雲端客戶端 ---
client = InferenceHTTPClient.init(
    api_url="https://serverless.roboflow.com",
    api_key=os.environ["ROBOFLOW_API_KEY"],
)

# --- 5. 逐段提交並處理 ---
for chunk_path in chunk_files:
    print(f"\n[開始處理片段] -> {chunk_path}")
    
    source = VideoFileSource(chunk_path, realtime_processing=False)
    config = StreamConfig(
        stream_output=[],
        data_output=DATA_OUTPUTS,
        requested_plan="webrtc-gpu-medium",
        requested_region="us",
    )
    
    session = client.webrtc.stream(
        source=source,
        workflow=ROBOFLOW_WORKFLOW,
        workspace=ROBOFLOW_WORKSPACE,
        image_input="image",
        config=config,
    )
    
    max_chunk_frame_id = 0
    max_chunk_pts_ms = 0.0
    # 本段暫存 buffer（在 callback 執行緒中累積，run() 結束後再批次寫入 DB）
    frame_buffer = []  # (frame_id, pts_ms, predictions_json)

    @session.on_data()
    def on_data(data: dict, metadata: VideoMetadata):
        global max_chunk_frame_id, max_chunk_pts_ms, debug_samples_left
        
        # 計算全局影格 ID 與時間戳（metadata 欄位缺漏時保護）
        frame_id = getattr(metadata, "frame_id", None)
        pts = getattr(metadata, "pts", None)
        time_base = getattr(metadata, "time_base", None)
        chunk_pts_ms = (pts * time_base * 1000) if (pts is not None and time_base) else 0.0
        local_frame = frame_id if frame_id is not None else (max_chunk_frame_id + 1)
        
        current_global_frame = global_frame_offset + local_frame
        current_global_pts = global_time_offset_ms + chunk_pts_ms
        if local_frame > max_chunk_frame_id:
            max_chunk_frame_id = local_frame
        if chunk_pts_ms > max_chunk_pts_ms:
            max_chunk_pts_ms = chunk_pts_ms
        
        # 原樣保存 predictions（不解析、不假設結構）
        raw = data.get("predictions") if isinstance(data, dict) else None
        frame_buffer.append((current_global_frame, current_global_pts, json.dumps(raw)))
        
        # 診斷：印出前幾個非空 payload 的真實結構，方便確認欄位名
        if debug_samples_left > 0 and raw:
            keys = list(data.keys()) if isinstance(data, dict) else type(data).__name__
            print(f"[payload 樣本] data.keys={keys} | predictions type={type(raw).__name__} "
                  f"| repr={repr(raw)[:500]}")
            debug_samples_left -= 1

    session.run()
    
    cur.executemany("INSERT OR REPLACE INTO frames VALUES (?,?,?)", frame_buffer)
    conn.commit()
    print(f"[片段寫入完成] 影格 {len(frame_buffer)} 筆")
    
    # 更新下一段影片的全局偏移量
    global_frame_offset += (max_chunk_frame_id + 1)
    global_time_offset_ms += max_chunk_pts_ms
    
    try:
        os.remove(chunk_path)
    except OSError:
        pass

# 寫入後統計含標註的影格數，立即得知是否真的取得 predictions
n_total, n_nonempty = cur.execute(
    "SELECT COUNT(*), SUM(CASE WHEN predictions_json NOT IN ('null','[]') THEN 1 ELSE 0 END) FROM frames"
).fetchone()
conn.close()

print(f"\n=== 步驟一完成 ===")
print(f"總影格 {n_total} 筆，其中含標註 {n_nonempty or 0} 筆 -> {os.path.abspath(DB_PATH)}")
if not n_nonempty:
    print("⚠️ 所有 predictions 皆為空：請檢查上方 payload 樣本的 data.keys()，"
          "確認 DATA_OUTPUTS 的輸出名稱是否正確。")


In [ ]:
# 從 SQLite 解析標註、計數並重畫到原影片，輸出 labeled.mp4
import cv2
import json
import sqlite3
from collections import defaultdict

INPUT_VIDEO = "origin.mp4"
DB_PATH = "roboflow_data.db"
OUTPUT_VIDEO = "labeled.mp4"
COUNT_LINE_Y = 120       # 計數警戒線 y
COUNT_LINE_X_MIN = 0     # 計數線段 x 起點
COUNT_LINE_X_MAX = 160   # 計數線段 x 終點

def parse_predictions(raw):
    """把步驟一原樣保存的 predictions 解析成 list[dict]，容忍多種結構。"""
    if isinstance(raw, str):
        try:
            raw = json.loads(raw)
        except json.JSONDecodeError:
            return []
    if raw is None:
        return []
    if isinstance(raw, dict):
        # 常見：{"predictions": [...]}；否則找出第一個 list 值
        if isinstance(raw.get("predictions"), list):
            return raw["predictions"]
        for v in raw.values():
            if isinstance(v, list):
                return v
        return [raw]  # 單一物件 dict
    if isinstance(raw, list):
        return raw
    return []

# --- 1. 讀取所有影格的原始標註 ---
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

rows = cur.execute(
    "SELECT frame_id, pts_ms, predictions_json FROM frames ORDER BY frame_id"
).fetchall()
preds_by_frame = {fid: (pts, parse_predictions(pj)) for fid, pts, pj in rows}

# 診斷：印出第一個非空樣本的結構，確認欄位名
for fid, (pts, plist) in preds_by_frame.items():
    if plist:
        print(f"[解析樣本] frame {fid}: {len(plist)} 筆，第一筆 keys="
              f"{list(plist[0].keys()) if isinstance(plist[0], dict) else type(plist[0]).__name__}")
        break
else:
    print("⚠️ frames 表中沒有任何非空 predictions，請先檢查步驟一輸出。")

# --- 2. 依 frame_id 順序偵測跨線並累計去重計數，並把計數落地 frame_counts ---
cur.executescript("""
DROP TABLE IF EXISTS frame_counts;
CREATE TABLE frame_counts (
    frame_id    INTEGER PRIMARY KEY,
    pts_ms      REAL,
    counts_json TEXT
);
""")

class_ids = defaultdict(set)  # 各類別已跨線(計數過)的 tracker_id
prev_pos = {}                 # tracker_id -> 上一次出現的中心 (x, y)
count_rows = []
for fid in sorted(preds_by_frame):
    pts, plist = preds_by_frame[fid]
    for pred in plist:
        if not isinstance(pred, dict):
            continue
        track_id = pred.get("tracker_id")
        vtype = pred.get("class")
        x = pred.get("x")
        y = pred.get("y")
        if track_id is None or vtype is None or x is None or y is None:
            continue
        prev = prev_pos.get(track_id)
        prev_pos[track_id] = (x, y)   # 不論是否計數，都更新軌跡上一點
        if prev is None:              # 第一次看到此 track，無法判定跨線
            continue
        if any(track_id in ids for ids in class_ids.values()):
            continue                  # 已計數過
        px, py = prev
        # 前後兩次出現分屬計數線兩側 -> 連線穿過 y=COUNT_LINE_Y（雙向）
        if (py - COUNT_LINE_Y) * (y - COUNT_LINE_Y) < 0:
            # 連線與 y=COUNT_LINE_Y 的交點 x（y != py 已由上式保證）
            t = (COUNT_LINE_Y - py) / (y - py)
            cross_x = px + t * (x - px)
            if COUNT_LINE_X_MIN <= cross_x <= COUNT_LINE_X_MAX:
                class_ids[vtype].add(track_id)
    snapshot = {cls: len(ids) for cls, ids in class_ids.items()}
    count_rows.append((fid, pts, json.dumps(snapshot)))

cur.executemany("INSERT OR REPLACE INTO frame_counts VALUES (?,?,?)", count_rows)
conn.commit()

final_total = sum(len(ids) for ids in class_ids.values())
summary = ", ".join(f"{cls}={len(ids)}" for cls, ids in sorted(class_ids.items()))
print(f"=== 計數完成 === Total={final_total}, {summary}")

# 把每幀累計計數讀回記憶體供繪製
counts_by_frame = {fid: json.loads(cj) for fid, _, cj in count_rows}

# --- 3. 開原影片，逐幀重畫框 + 計數 + 計數線 ---
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    conn.close()
    raise RuntimeError(f"無法開啟原影片：{INPUT_VIDEO}")

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
print(f"開始繪製：{INPUT_VIDEO} ({w}x{h} @ {fps:.1f}fps) -> {OUTPUT_VIDEO}")

AQUA = (255, 255, 0)  # 青色
RED = (0, 0, 255)      # 紅色
YELLOW = (0, 255, 255)   # 黃色
FONT = cv2.FONT_HERSHEY_SIMPLEX
last_counts = {}        # 延續上一筆計數，避免無資料的影格閃爍
idx = 0
written = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 重畫偵測框（中心座標 -> 角點）
    _, plist = preds_by_frame.get(idx, (None, []))
    for pred in plist:
        if not isinstance(pred, dict):
            continue
        x, y = pred.get("x"), pred.get("y")
        bw, bh = pred.get("width"), pred.get("height")
        if x is None or bw is None:
            continue
        x1, y1 = int(x - bw / 2), int(y - bh / 2)
        x2, y2 = int(x + bw / 2), int(y + bh / 2)
        cv2.rectangle(frame, (x1, y1), (x2, y2), RED, 1)
        label = f"{pred.get('class', '')}"
        cv2.putText(frame, label, (x1, max(y1 - 4, 10)), FONT, 0.4, RED, 1)

    # 累計計數（無則沿用上一筆）
    if idx in counts_by_frame:
        last_counts = counts_by_frame[idx]
    counts = last_counts

    # 計數警戒線（線段 x=0~160, y=100）
    cv2.line(frame, (COUNT_LINE_X_MIN, COUNT_LINE_Y),
             (COUNT_LINE_X_MAX, COUNT_LINE_Y), AQUA, 1)

    # 左上角總計與各類別
    total = sum(counts.values())
    cv2.putText(frame, f"Total: {total}", (10, 20), FONT, 0.5, YELLOW, 1)
    ty = 38
    for cls_name, count in sorted(counts.items()):
        cv2.putText(frame, f"{cls_name}: {count}", (10, ty), FONT, 0.5, YELLOW, 1)
        ty += 18

    out.write(frame)
    written += 1
    idx += 1

cap.release()
out.release()
conn.close()
print(f"完成：共輸出 {written} 個影格 -> {OUTPUT_VIDEO}")
